# セミナーII 第2回 実践：ViT で分類して、ResNet-18 と比べる

第1回では、CNN（ResNet-18）の分類ヘッドを37品種用に付け替えて、犬と猫の品種を分類しました。ここでは、モデルを **Transformer（ViT-B/16）** に変えて、同じデータ・同じ学習の進め方で学習し、test の正解率と時間を比べます。

ViT は、画像を 16×16 画素のパッチに分けてトークンの列にし、先頭に **CLS トークン**を足して Transformer で計算します。CLS トークンの出力（768次元）に分類ヘッドを付ければ、分類モデルになります（スライド「ViT で画像を分類する（CLS トークン）」）。

**最初に**

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します（セルの左の ▶、または **Shift + Enter**）。

ログインや Google ドライブへのアクセスの許可は必要ありません。全体で約20分（うち計算 約10分）です。

- データの用意と学習のコードは、第1回と同じもの（`lesson01/` の `data.py`、`engine.py`、`models.py`）を使います。
- しばらく操作しないと接続が切れます。`NameError` が出たときは、1. から順に実行し直してください。

## 0. このノートブックの流れ

目安：約1分

| 節 | やること |
| --- | --- |
| 1. 準備 | 第1回のコードとライブラリを読み込む |
| 2. 2つのモデルの違い | ResNet-18 と ViT-B/16 で、分類ヘッドに入るベクトルを確かめる |
| 3. 学習して比べる | 同じデータで学習し、test の正解率と時間を比べる |
| 4. やってみよう | 学習率を変える、学習済みの重みを使わない |
| 5. まとめ | 分かったこと |

## 1. 準備

目安：約3分（うち計算 約1分）

公開リポジトリから、第1回のコード（`lesson01/` の `.py`）を取得して、import できるようにします。`device: cuda` と GPU の名前が表示されれば準備完了です。`device: cpu` と表示されたら、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください。

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/r-terasawa457/seminar2-2026.git"
REPO_REF = "main"  # 授業当日は、教材が変わらないようタグを指定する

if Path("../lesson01/data.py").exists():  # リポジトリの lesson02 で開いたとき
    LESSON01 = Path("../lesson01")
else:  # Colab では、最初はファイルが無いので取得する
    if not Path("seminar2-2026").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "seminar2-2026"],
                       check=True)
    LESSON01 = Path("seminar2-2026/lesson01")
# import で第1回の .py を見つけられるよう、探す場所に加える
sys.path.insert(0, str(LESSON01.resolve()))

# Colab に入っていないライブラリがあれば入れる
for package in ["timm", "datasets"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import matplotlib.pyplot as plt
import pandas as pd
import timm
import torch
import torch.nn as nn

from data import build_transforms, load_splits, make_loader  # 第1回の 3.（データを用意する）
from engine import evaluate, fit                              # 第1回の 6.、7.（学習、test）
from models import build_model                                # 第1回の 4.（モデル）

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 2 if device == "cuda" and os.name != "nt" else 0  # データを読む並列数
print("第1回のコード:", LESSON01.resolve())
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

第1回と同じ Oxford-IIIT Pet（37品種）を、同じ分け方で train / validation / test に分けます。

In [ ]:
SEED = 0
train_data, val_data, test_data, class_names = load_splits(SEED)
num_classes = len(class_names)
print("train / validation / test:", len(train_data), len(val_data), len(test_data), "| クラス数:", num_classes)

## 2. 2つのモデルの違い

目安：約5分（うち計算 約1分）

どちらも timm の学習済みモデルで、第1回と同じ `build_model` で分類ヘッドを37品種用に付け替えます。初回は重み（ViT-B/16 は約350MB）をダウンロードします。`HF_TOKEN` に関する警告が表示されても無視してかまいません。

- ResNet-18：[timm/resnet18.a1_in1k](https://huggingface.co/timm/resnet18.a1_in1k)（Apache-2.0）。ImageNet-1k で学習
- ViT-B/16：[timm/vit_base_patch16_224.augreg2_in21k_ft_in1k](https://huggingface.co/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k)（Apache-2.0）。ImageNet-21k（約1400万枚）で学習し、ImageNet-1k で追加学習
- 論文：A. Dosovitskiy et al., "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale," ICLR 2021（[arXiv:2010.11929](https://arxiv.org/abs/2010.11929)）

In [ ]:
RESNET = "resnet18.a1_in1k"
VIT = "vit_base_patch16_224.augreg2_in21k_ft_in1k"

resnet = build_model(RESNET, num_classes).eval()
vit = build_model(VIT, num_classes).eval()
for name, m in [("ResNet-18", resnet), ("ViT-B/16", vit)]:
    print(f"{name}: パラメータ {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M 個 | 分類ヘッド {m.get_classifier()}")

1枚の画像（224×224）を入れて、分類ヘッドの手前までの形を確かめます。

- ResNet-18：最後の特徴マップ（512チャンネル × 7×7）を、位置について平均して、512次元のベクトルにする（第1回）
- ViT-B/16：224÷16 = 14 なので、14×14 = 196個のパッチのトークンに、CLS トークンを1個足した197個のトークン（各768次元）を12層の Transformer で計算する。分類ヘッドには、**CLS トークンの出力**（768次元）だけを入れる

In [ ]:
_, eval_transform = build_transforms()
x = eval_transform(test_data[0]["image"].convert("RGB")).unsqueeze(0)  # (1, 3, 224, 224)

with torch.no_grad():
    feature_map = resnet.forward_features(x)                     # 特徴マップ
    resnet_vec = resnet.forward_head(feature_map, pre_logits=True)  # 位置について平均
    tokens = vit.forward_features(x)                             # すべてのトークンの出力
    vit_vec = vit.forward_head(tokens, pre_logits=True)          # CLS トークン（先頭）の出力

print("入力                    ", tuple(x.shape))
print("ResNet-18 特徴マップ    ", tuple(feature_map.shape), "→ 平均 →", tuple(resnet_vec.shape),
      "→ 分類ヘッド →", tuple(resnet.get_classifier()(resnet_vec).shape))
print("ViT-B/16  トークンの出力", tuple(tokens.shape), "→ CLS  →", tuple(vit_vec.shape),
      "→ 分類ヘッド →", tuple(vit.get_classifier()(vit_vec).shape))
print("CLS トークンの出力と同じか:", torch.equal(vit_vec, tokens[:, 0]))

前処理の平均と標準偏差も、モデルが学習したときの値に合わせます（第1回の 3-3）。ResNet-18 は ImageNet の値、この ViT は 0.5 です。

In [ ]:
for name, m in [("ResNet-18", resnet), ("ViT-B/16", vit)]:
    config = timm.data.resolve_model_data_config(m)
    print(f"{name}: 入力 {config['input_size']} | 平均 {config['mean']} | 標準偏差 {config['std']}")
del resnet, vit

## 3. 学習して比べる

目安：約10分（うち計算 約8分。ResNet-18 が約1.5分、ViT-B/16 はその約3倍）

`run` は、第1回の 9. の `run_experiment` と同じく、データの用意から test までを通して実行し、結果を `results` に加えます。どちらも 3 epoch 学習します。

ViT-B/16 は、設定を2つ変えます。

- **学習率を 0.0001 にする**：第1回の 0.001 では、ViT は学習が進みません（4. で試せます）。大きなモデルを追加学習するときは、学習率を小さくするのが一般的です。
- **batch サイズを 32 にする**：ViT-B/16 は ResNet-18 よりパラメータも計算も多く、64枚では GPU のメモリが足りなくなることがあります。

In [ ]:
EPOCHS = 3
results = []


def run(note, model_name, learning_rate, batch_size, pretrained=True, epochs=EPOCHS):
    # 第1回の 9. の run_experiment と同じ流れ。前処理の平均と標準偏差はモデルに合わせる
    torch.manual_seed(SEED)
    model = build_model(model_name, num_classes, pretrained=pretrained).to(device)
    config = timm.data.resolve_model_data_config(model)
    train_transform, eval_transform = build_transforms(mean=config["mean"], std=config["std"])
    train_loader = make_loader(train_data, train_transform, batch_size, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = make_loader(val_data, eval_transform, batch_size, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = make_loader(test_data, eval_transform, batch_size, shuffle=False, num_workers=NUM_WORKERS)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    start = time.time()
    history, best_acc = fit(model, train_loader, val_loader, criterion, optimizer, device, epochs)
    minutes = (time.time() - start) / 60
    _, test_acc, _, _ = evaluate(model, test_loader, criterion, device)
    print(f"best val acc {best_acc:.3f}  |  test acc {test_acc:.3f}  |  {minutes:.1f} 分")
    results.append({"note": note, "lr": learning_rate, "batch_size": batch_size, "pretrained": pretrained,
                    "test_acc": test_acc, "minutes": minutes, "history": history})
    del model
    torch.cuda.empty_cache()


def show_results():
    table = pd.DataFrame([{"": r["note"], "学習率": r["lr"], "batch": r["batch_size"], "学習済み": r["pretrained"],
                           "test の正解率": round(r["test_acc"], 3), "時間（分）": round(r["minutes"], 1)}
                          for r in results])
    display(table.set_index(""))

In [ ]:
run("ResNet-18", RESNET, learning_rate=1e-3, batch_size=64)  # 第1回の本編と同じ設定

In [ ]:
run("ViT-B/16", VIT, learning_rate=1e-4, batch_size=32)

In [ ]:
show_results()

plt.figure(figsize=(5, 3.2))
for r in results:
    plt.plot(range(1, len(r["history"]["val_acc"]) + 1), r["history"]["val_acc"], marker="o", label=r["note"])
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

同じ 3 epoch で、ViT-B/16 のほうが test の正解率が少し高くなります。代わりに、時間は約3倍かかります。手元の GPU（RTX 4060）では、ResNet-18 が 0.82（約1分）、ViT-B/16 が 0.85（約3.4分）でした。同じ設定でも、正解率は実行のたびに数ポイント変わります（第1回の 7-2）。

グラフを見ると、ViT-B/16 は 1 epoch 目で validation の正解率が 0.9 近くになり、その後は少し下がりました（train の正解率は上がり続けるので、学習しすぎ）。`fit` は validation の正解率が一番良かった時点の重みを残すので（第1回の 6.）、test はその重みで測っています。

ViT-B/16 が強いのは、Transformer だからというだけでなく、約1400万枚（ImageNet-21k）で事前に学習してあるためでもあります（スライド「④ データを増やすほど伸びる。代わりに苦手なこと」）。4. で、学習済みの重みを使わない場合と比べてみましょう。

## 4. やってみよう

目安：1回あたり約2〜5分（計算）

下のセルを書き換えて実行し、最後に `show_results()` で表を見比べましょう。

1. **ViT の学習率を、第1回と同じ 0.001 にする**：`run("ViT 学習率0.001", VIT, learning_rate=1e-3, batch_size=32)`。正解率はどうなるか
2. **学習済みの重みを使わない**（一から学習する）：`run("ResNet 一から", RESNET, learning_rate=1e-3, batch_size=64, pretrained=False)` と `run("ViT 一から", VIT, learning_rate=1e-4, batch_size=32, pretrained=False)`。3000枚ほどの画像で一から学ぶと、どちらが苦手か

In [ ]:
run("ViT 学習率0.001", VIT, learning_rate=1e-3, batch_size=32)
show_results()

手元の GPU（RTX 4060、3 epoch）での結果です。試してから見比べてください。

- ViT の学習率 0.001：test 0.04。37品種からでたらめに選ぶ（約0.03）のとほぼ同じで、学習が進んでいません。
- 一から学習：ResNet-18 が 0.12、ViT-B/16 が 0.06。どちらも学習済みの重みから始めた場合（0.82、0.85）に遠く及ばず、ViT のほうが低くなりました。3000枚ほどのデータと 3 epoch では、一から学ぶには足りません。

## 5. まとめ

目安：約1分

| 分かったこと | 節 |
| --- | --- |
| ViT は、CLS トークンの出力（768次元）に分類ヘッドを付けて分類する。ResNet-18 は、特徴マップを平均した512次元のベクトルに付ける | 2. |
| 同じデータ・同じ学習の進め方で、モデルの名前を変えるだけで比べられる | 3. |
| 学習済みの ViT-B/16 は、ResNet-18 より正解率が少し高い。代わりに時間は約3倍 | 3. |
| 大きなモデルを追加学習するときは、学習率を小さくする | 4. |
| 少ないデータで一から学ぶと、どちらも大きく下がり、ViT のほうが低い。大量のデータで事前に学習した重みが効いている | 4. |

研究でも、timm などのライブラリのモデル名を変えて、CNN と Transformer を同じ条件で比べることがよくあります。そのときは、学習率や batch サイズ、前処理をモデルに合わせて調整します。